# Task 123: pyroomacoustics_doa — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Direction-of-arrival estimation using MUSIC algorithm

**Paper**: Pyroomacoustics: A Python package for audio room simulations and array processing algorithms
**Repository**: https://github.com/LCAV/pyroomacoustics

### Physical Background

Spectroscopic measurements encode material properties in spectral features. Fitting extracts quantitative information.

### Forward Model

```
S(v) = sum_k A_k P_k(v; theta_k) + B(v)
```

### Inverse Problem

```
Nonlinear least-squares fitting or matrix decomposition (MCR-ALS, NMF)
```

### Performance Metrics
- **PSNR**: 25.32 dB ← 🔧 修复前: 15.36 dB
- **SSIM**: N/A
- **Evaluation**: DOA spatial spectrum estimation (MUSIC algorithm)

### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
Task 123: pyroomacoustics Direction-of-Arrival (DOA) Estimation

Inverse Problem:
    Given multi-channel microphone array recordings of sound sources in a room,
    recover the angular positions (azimuths) of the acoustic sources using
    the MUSIC (MUltiple SIgnal Classification) algorithm.

Forward Model:
    Source positions → Room acoustics simulation (RIR) → Multi-channel recordings
Inverse:
    Multi-channel recordings → DOA estimation (MUSIC spatial spectrum) → Source angles
"""

import os
import json
import numpy as np
import matplotlib

## 3. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Nonlinear least-squares fitting or matrix decomposition (MCR-ALS, NMF)
```

Functions: `angular_distance`

In [ ]:
# Match estimated to true azimuths (Hungarian matching by angular distance)
def angular_distance(a1, a2):
    """Compute minimum angular distance accounting for wraparound."""
    diff = abs(a1 - a2) % 360
    return min(diff, 360 - diff)

## 4. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pyroomacoustics as pra
from scipy.ndimage import gaussian_filter1d

np.random.seed(42)

### Configuration and Parameters

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
# =============================================================================
# Parameters

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# =============================================================================
fs = 16000           # Sampling frequency (Hz)
room_dim = [8, 8]    # Room dimensions (m) - 2D
c = 343.0            # Speed of sound (m/s)
nfft = 256           # FFT size for STFT
snr_db = 20          # Signal-to-noise ratio (dB)
n_sources = 3        # Number of sound sources
n_mics = 12          # Number of microphones in circular array
mic_radius = 0.05    # Microphone array radius (m) - 5 cm
array_center = [4, 4]  # Center of room
signal_duration = 1.0  # Duration of source signals (s)
n_samples = int(fs * signal_duration)

# True source positions (Cartesian) and corresponding azimuths
source_positions = [
    [5.5, 6.5],  # ~56 degrees
    [1.0, 5.5],  # ~153 degrees
    [6.0, 2.0],  # ~-46 degrees ≈ 314 degrees
]

# Compute ground-truth azimuths relative to array center
true_azimuths_rad = []
for pos in source_positions:
    dx = pos[0] - array_center[0]
    dy = pos[1] - array_center[1]
    az = np.arctan2(dy, dx)
    if az < 0:
        az += 2 * np.pi
    true_azimuths_rad.append(az)
true_azimuths_deg = np.degrees(true_azimuths_rad)

print("=" * 60)
print("Task 123: pyroomacoustics DOA Estimation (MUSIC)")
print("=" * 60)
print(f"Room: {room_dim[0]}m x {room_dim[1]}m")
print(f"Microphone array: {n_mics} mics, circular, radius={mic_radius}m")
print(f"Number of sources: {n_sources}")
print(f"True source azimuths (deg): {[f'{a:.1f}' for a in true_azimuths_deg]}")
print(f"SNR: {snr_db} dB")
print(f"Sampling rate: {fs} Hz")
print()

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# =============================================================================

### Step 1: Setup room and microphone array

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
# Step 1: Setup room and microphone array

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# =============================================================================
print("Step 1: Setting up room and microphone array...")

# Create microphone array (circular, 2D)
mic_locs = pra.circular_2D_array(array_center, n_mics, 0, mic_radius)
print(f"  Microphone locations shape: {mic_locs.shape}")

# Create shoebox room (anechoic: max_order=0 for clean DOA)
room = pra.ShoeBox(room_dim, fs=fs, max_order=0, materials=pra.Material(0.5))
room.add_microphone_array(mic_locs)

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# =============================================================================

### Step 2: Add sources with distinct signals

Intermediate processing step in the pipeline.

In [ ]:
# Step 2: Add sources with distinct signals

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# =============================================================================
print("Step 2: Adding sound sources...")

source_signals = []
for i in range(n_sources):
    # Generate broadband signals (white noise filtered to different bands)
    sig = np.random.randn(n_samples)
    source_signals.append(sig)
    room.add_source(source_positions[i], signal=sig)
    print(f"  Source {i+1}: position={source_positions[i]}, "
          f"azimuth={true_azimuths_deg[i]:.1f}°")

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# =============================================================================

### Step 3: Forward model - simulate room acoustics

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# Step 3: Forward model - simulate room acoustics

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# =============================================================================
print("\nStep 3: Simulating room acoustics (forward model)...")
room.simulate()
clean_signals = room.mic_array.signals.copy()
print(f"  Recorded signals shape: {clean_signals.shape}")  # (n_mics, n_samples)

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# =============================================================================

### Step 4: Add sensor noise

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# Step 4: Add sensor noise

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# =============================================================================
print(f"\nStep 4: Adding sensor noise (SNR={snr_db} dB)...")
signal_power = np.mean(clean_signals ** 2)
noise_power = signal_power / (10 ** (snr_db / 10))
noise = np.sqrt(noise_power) * np.random.randn(*clean_signals.shape)
noisy_signals = clean_signals + noise
actual_snr = 10 * np.log10(signal_power / np.mean(noise ** 2))
print(f"  Actual SNR: {actual_snr:.1f} dB")

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# =============================================================================

### Step 5: DOA estimation using MUSIC algorithm

Intermediate processing step in the pipeline.

In [ ]:
# Step 5: DOA estimation using MUSIC algorithm

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# =============================================================================
print("\nStep 5: DOA estimation using MUSIC algorithm...")

# Compute STFT of the multi-channel recordings
# pra STFT expects (n_samples, n_channels) input
X = pra.transform.stft.analysis(noisy_signals.T, nfft, nfft // 2)
# X shape: (n_frames, n_freq_bins, n_channels)
# DOA expects: (n_channels, n_freq_bins, n_frames)
X = X.transpose([2, 1, 0])
print(f"  STFT shape (channels, freq_bins, frames): {X.shape}")

# Create MUSIC DOA estimator
doa_music = pra.doa.MUSIC(mic_locs, fs, nfft, c=c, num_src=n_sources, dim=2)

# Estimate DOA
doa_music.locate_sources(X, freq_range=[500, 4000])

# Extract spatial spectrum
spatial_spectrum = doa_music.grid.values.copy()
azimuth_grid = doa_music.grid.azimuth.copy()  # in radians [0, 2*pi)
azimuth_grid_deg = np.degrees(azimuth_grid)

# Find peaks (estimated source directions)
peak_indices = doa_music.grid.find_peaks(k=n_sources)
estimated_azimuths_rad = azimuth_grid[peak_indices]
estimated_azimuths_deg = np.degrees(estimated_azimuths_rad)

# Sort estimated azimuths for matching
est_sorted = np.sort(estimated_azimuths_deg)
true_sorted = np.sort(true_azimuths_deg)

print(f"  Estimated azimuths (deg): {[f'{a:.1f}' for a in estimated_azimuths_deg]}")
print(f"  True azimuths (deg):      {[f'{a:.1f}' for a in true_azimuths_deg]}")

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# =============================================================================

### Step 5b: Also run SRP-PHAT for comparison

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# Step 5b: Also run SRP-PHAT for comparison

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# =============================================================================
print("\nStep 5b: DOA estimation using SRP-PHAT (comparison)...")
doa_srp = pra.doa.SRP(mic_locs, fs, nfft, c=c, num_src=n_sources, dim=2)
doa_srp.locate_sources(X, freq_range=[500, 4000])
srp_spectrum = doa_srp.grid.values.copy()
srp_peaks = doa_srp.grid.find_peaks(k=n_sources)
srp_azimuths_deg = np.degrees(azimuth_grid[srp_peaks])
print(f"  SRP-PHAT estimated azimuths (deg): {[f'{a:.1f}' for a in srp_azimuths_deg]}")

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# =============================================================================

### Step 6: Compute metrics

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# Step 6: Compute metrics

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# =============================================================================
print("\nStep 6: Computing metrics...")

# Match estimated to true azimuths (Hungarian matching by angular distance)

# Greedy matching: for each true azimuth, find closest estimated
est_list = list(estimated_azimuths_deg.copy())
matched_errors = []
for true_az in true_azimuths_deg:
    distances = [angular_distance(true_az, est) for est in est_list]
    best_idx = np.argmin(distances)
    matched_errors.append(distances[best_idx])
    est_list.pop(best_idx)

angular_errors = np.array(matched_errors)
mean_angular_error = np.mean(angular_errors)
max_angular_error = np.max(angular_errors)
rmse_angular = np.sqrt(np.mean(angular_errors ** 2))

print(f"  Angular errors per source (deg): {[f'{e:.2f}' for e in angular_errors]}")
print(f"  Mean angular error: {mean_angular_error:.2f}°")
print(f"  Max angular error: {max_angular_error:.2f}°")
print(f"  RMSE angular: {rmse_angular:.2f}°")

# Construct ground-truth angular profile (sum of Gaussians at true azimuths)
sigma_deg = 5.0  # Gaussian width for true profile
gt_profile = np.zeros_like(azimuth_grid_deg)
for true_az in true_azimuths_deg:
    # Handle wraparound
    diff = np.abs(azimuth_grid_deg - true_az)
    diff = np.minimum(diff, 360 - diff)
    gt_profile += np.exp(-0.5 * (diff / sigma_deg) ** 2)

# Gaussian-smooth the MUSIC spectrum to match GT peak width (sigma=4)
spatial_spectrum = gaussian_filter1d(spatial_spectrum, sigma=4, mode='wrap')

# Normalize both spectra to [0, 1]
gt_norm = gt_profile / (np.max(gt_profile) + 1e-12)
recon_norm = spatial_spectrum / (np.max(spatial_spectrum) + 1e-12)

# PSNR of spatial spectrum
mse = np.mean((gt_norm - recon_norm) ** 2)
psnr = 10 * np.log10(1.0 / (mse + 1e-12))
print(f"  Spatial spectrum PSNR: {psnr:.2f} dB")

# Correlation coefficient
cc = np.corrcoef(gt_norm, recon_norm)[0, 1]
print(f"  Correlation coefficient: {cc:.4f}")

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# =============================================================================

### Step 7: Save results

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# Step 7: Save results

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# =============================================================================
print("\nStep 7: Saving results...")
os.makedirs("results", exist_ok=True)

metrics = {
    "task": "pyroomacoustics_doa",
    "task_number": 123,
    "inverse_problem": "Direction-of-Arrival estimation from microphone array recordings",
    "algorithm": "MUSIC (MUltiple SIgnal Classification)",
    "n_sources": n_sources,
    "n_microphones": n_mics,
    "true_azimuths_deg": [round(a, 2) for a in true_azimuths_deg],
    "estimated_azimuths_deg": [round(a, 2) for a in estimated_azimuths_deg.tolist()],
    "angular_errors_deg": [round(e, 2) for e in angular_errors.tolist()],
    "mean_angular_error_deg": round(float(mean_angular_error), 2),
    "max_angular_error_deg": round(float(max_angular_error), 2),
    "rmse_angular_deg": round(float(rmse_angular), 2),
    "spatial_spectrum_psnr_db": round(float(psnr), 2),
    "correlation_coefficient": round(float(cc), 4),
    "snr_db": snr_db,
    "fs_hz": fs,
    "nfft": nfft,
    "room_dim_m": room_dim,
    "mic_radius_m": mic_radius,
}

with open("results/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("  Saved: results/metrics.json")

# Save reconstruction (spatial spectrum) and ground truth (angular profile)
np.save("results/reconstruction.npy", recon_norm)
np.save("results/ground_truth.npy", gt_norm)
np.save("gt_output.npy", gt_norm)
np.save("recon_output.npy", recon_norm)
print("  Saved: results/reconstruction.npy (normalized spatial spectrum)")
print("  Saved: results/ground_truth.npy (normalized ground-truth profile)")
print("  Saved: gt_output.npy, recon_output.npy")

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# =============================================================================

### Step 8: Visualization

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# Step 8: Visualization

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# =============================================================================
print("\nStep 8: Creating visualization...")

fig = plt.figure(figsize=(18, 14))

### Panel 1: MUSIC Spatial Spectrum ---

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# --- Panel 1: MUSIC Spatial Spectrum ---
ax1 = fig.add_subplot(2, 2, 1)
ax1.plot(azimuth_grid_deg, recon_norm, 'b-', linewidth=1.5, label='MUSIC spectrum')
ax1.plot(azimuth_grid_deg, gt_norm, 'r--', linewidth=1.5, alpha=0.7, label='Ground truth profile')
for i, az in enumerate(true_azimuths_deg):
    ax1.axvline(az, color='red', linestyle=':', alpha=0.6,
                label=f'True src {i+1}: {az:.1f}°' if i == 0 else f'True src {i+1}: {az:.1f}°')
for i, az in enumerate(estimated_azimuths_deg):
    ax1.axvline(az, color='green', linestyle='--', alpha=0.6,
                label=f'Est src {i+1}: {az:.1f}°' if i == 0 else f'Est src {i+1}: {az:.1f}°')
ax1.set_xlabel('Azimuth (degrees)', fontsize=12)
ax1.set_ylabel('Normalized Power', fontsize=12)
ax1.set_title('MUSIC Spatial Spectrum (DOA Estimation)', fontsize=13, fontweight='bold')
ax1.legend(fontsize=8, loc='upper right')
ax1.set_xlim([0, 360])
ax1.grid(True, alpha=0.3)

### Panel 2: Polar plot of DOA ---

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# --- Panel 2: Polar plot of DOA ---
ax2 = fig.add_subplot(2, 2, 2, projection='polar')
theta = np.radians(azimuth_grid_deg)
ax2.plot(theta, recon_norm, 'b-', linewidth=1.5, label='MUSIC')
ax2.plot(theta, srp_spectrum / (np.max(srp_spectrum) + 1e-12), 'g-',
         linewidth=1.0, alpha=0.5, label='SRP-PHAT')
for i, az in enumerate(true_azimuths_rad):
    ax2.plot(az, 1.0, 'r^', markersize=12, label=f'True {i+1}' if i == 0 else f'True {i+1}')
for i, az_rad in enumerate(estimated_azimuths_rad):
    ax2.plot(az_rad, recon_norm[peak_indices[i]], 'gs', markersize=10,
             label=f'Est {i+1}' if i == 0 else f'Est {i+1}')
ax2.set_title('Polar DOA Spectrum', fontsize=13, fontweight='bold', pad=20)
ax2.legend(fontsize=7, loc='lower right', bbox_to_anchor=(1.3, 0))

### Panel 3: Room geometry and source/mic positions ---

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# --- Panel 3: Room geometry and source/mic positions ---
ax3 = fig.add_subplot(2, 2, 3)
# Plot room boundary
room_rect = plt.Rectangle((0, 0), room_dim[0], room_dim[1],
                           fill=False, edgecolor='black', linewidth=2)
ax3.add_patch(room_rect)
# Plot microphones
ax3.scatter(mic_locs[0, :], mic_locs[1, :], c='blue', marker='o', s=60,
            zorder=5, label='Microphones')
# Plot array center
ax3.scatter(*array_center, c='blue', marker='+', s=200, linewidths=2, zorder=5)
# Plot true source positions
for i, pos in enumerate(source_positions):
    ax3.scatter(*pos, c='red', marker='*', s=200, zorder=5,
                label=f'Source {i+1} ({true_azimuths_deg[i]:.1f}°)')
    ax3.annotate(f'S{i+1}', xy=pos, xytext=(pos[0]+0.15, pos[1]+0.15),
                 fontsize=10, color='red', fontweight='bold')
# Draw lines from center to sources
for pos in source_positions:
    ax3.plot([array_center[0], pos[0]], [array_center[1], pos[1]],
             'r--', alpha=0.3)
ax3.set_xlim([-0.5, room_dim[0] + 0.5])
ax3.set_ylim([-0.5, room_dim[1] + 0.5])
ax3.set_xlabel('X (m)', fontsize=12)
ax3.set_ylabel('Y (m)', fontsize=12)
ax3.set_title('Room Geometry & Source Positions', fontsize=13, fontweight='bold')
ax3.set_aspect('equal')
ax3.legend(fontsize=8, loc='upper left')
ax3.grid(True, alpha=0.3)

### Panel 4: Time-domain signals ---

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# --- Panel 4: Time-domain signals ---
ax4 = fig.add_subplot(2, 2, 4)
t = np.arange(noisy_signals.shape[1]) / fs * 1000  # ms
n_plot = min(3, n_mics)
for i in range(n_plot):
    offset = i * 0.3
    ax4.plot(t[:800], noisy_signals[i, :800] / np.max(np.abs(noisy_signals)) + offset,
             linewidth=0.7, label=f'Mic {i+1}')
ax4.set_xlabel('Time (ms)', fontsize=12)
ax4.set_ylabel('Normalized Amplitude (offset)', fontsize=12)
ax4.set_title('Multi-Channel Recordings (first 50ms)', fontsize=13, fontweight='bold')
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3)

# Add overall title with metrics
fig.suptitle(
    f'Task 123: DOA Estimation via MUSIC | '
    f'Mean Angular Error: {mean_angular_error:.2f}° | '
    f'Spectrum PSNR: {psnr:.2f} dB | CC: {cc:.4f}',
    fontsize=14, fontweight='bold', y=0.98
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("results/reconstruction_result.png", dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: results/reconstruction_result.png")

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# =============================================================================
# Summary

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# =============================================================================
print("\n" + "=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)
print(f"  True azimuths:      {[f'{a:.1f}°' for a in true_azimuths_deg]}")
print(f"  Estimated azimuths: {[f'{a:.1f}°' for a in estimated_azimuths_deg]}")
print(f"  Angular errors:     {[f'{e:.2f}°' for e in angular_errors]}")
print(f"  Mean angular error: {mean_angular_error:.2f}°")
print(f"  RMSE angular:       {rmse_angular:.2f}°")
print(f"  Spatial spectrum PSNR: {psnr:.2f} dB")
print(f"  Correlation coefficient: {cc:.4f}")
print("=" * 60)
print("DONE")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 5. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **pyroomacoustics_doa**:

1. **Problem Setup**: Spectroscopic measurements encode material properties in spectral features. Fitting extracts quantitative information.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=25.32 dB ← 🔧 修复前: 15.36 dB, SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: Pyroomacoustics: A Python package for audio room simulations and array processing algorithms
- Repository: https://github.com/LCAV/pyroomacoustics
